<a href="https://colab.research.google.com/github/Jaydenkaleb/Predictive_Analysis_ML/blob/main/PA_Assignment_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Set up Apache Airflow**

In [1]:
!pip install apache-airflow

In [2]:
#Import libraries
import os
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [3]:
# Set Airflow home
os.environ["AIRFLOW_HOME"] = "/content/airflow"

# Create required directories
Path("/content/airflow/dags").mkdir(parents=True, exist_ok=True)
Path("/content/airflow/logs").mkdir(parents=True, exist_ok=True)

print("AIRFLOW_HOME =", os.environ["AIRFLOW_HOME"])

AIRFLOW_HOME = /content/airflow


In [4]:
# Check whether Airflow is installed in Colab.
%pip show apache-airflow

Name: apache-airflow
Version: 3.3.2
Summary: Programmatically author, schedule and monitor data pipelines
Home-page: https://airflow.apache.org/
Author: 
Author-email: Apache Software Foundation <dev@airflow.apache.org>
License: 
Location: /usr/local/lib/python3.13/dist-packages
Requires: apache-airflow-core, apache-airflow-task-sdk
Required-by: apache-airflow-providers-common-compat, apache-airflow-providers-common-io, apache-airflow-providers-common-sql, apache-airflow-providers-smtp, apache-airflow-providers-standard


In [12]:
%%writefile /content/airflow/dags/iris_preprocessing_dag.py

from datetime import datetime, timezone #set the DAG’s start date
from hashlib import sha256 #creates an identifier for each run’s output folder
from pathlib import Path #create folders and manage file paths.

from airflow.sdk import dag, task, get_current_context #defines the workflow

#Define the DAG
@dag(
    dag_id="iris_preprocessing",
    start_date=datetime(2026, 1, 1, tzinfo=timezone.utc),
    schedule=None,
    catchup=False,
    tags=["iris", "preprocessing"],
)
def iris_preprocessing():

    # Task 1: Load the Iris dataset.
    @task
    def load_data():
        import pandas as pd
        from sklearn.datasets import load_iris

        # Use a separate output folder for each DAG run.
        run_id = get_current_context()["run_id"]
        run_key = sha256(run_id.encode()).hexdigest()[:16]

        folder = Path("/content/airflow/data") / run_key
        folder.mkdir(parents=True, exist_ok=True)

        iris = load_iris()

        data = pd.DataFrame(
            iris.data,
            columns=[
                "sepal_length",
                "sepal_width",
                "petal_length",
                "petal_width",
            ],
        )
        data["target"] = iris.target

        raw_path = folder / "raw_iris.csv"
        data.to_csv(raw_path, index=False)

        print(f"Loaded {len(data)} rows.")
        return str(raw_path)

    # Task 2: Remove invalid entries.
    @task
    def clean_data(raw_path):
        import numpy as np
        import pandas as pd

        data = pd.read_csv(raw_path)
        original_rows = len(data)

        features = [
            "sepal_length",
            "sepal_width",
            "petal_length",
            "petal_width",
        ]

        # Convert nonnumeric values into missing values.
        for column in features + ["target"]:
            data[column] = pd.to_numeric(
                data[column], errors="coerce"
            )

        # Remove missing or infinite values.
        data = data.replace([np.inf, -np.inf], np.nan).dropna()

        # Keep positive measurements and valid class labels.
        valid_rows = (
            (data[features] > 0).all(axis=1)
            & data["target"].isin([0, 1, 2])
        )
        data = data.loc[valid_rows].copy()
        data["target"] = data["target"].astype(int)

        if data.empty:
            raise ValueError("No valid rows remain.")

        clean_path = Path(raw_path).with_name("clean_iris.csv")
        data.to_csv(clean_path, index=False)

        print(f"Removed {original_rows - len(data)} invalid rows.")
        return str(clean_path)

    # Task 3: Split, normalize features, and save CSV files.
    @task(multiple_outputs=False)
    def split_normalize_save(clean_path):
        import joblib
        import pandas as pd
        from sklearn.model_selection import train_test_split
        from sklearn.preprocessing import StandardScaler

        data = pd.read_csv(clean_path)
        features = [
            "sepal_length",
            "sepal_width",
            "petal_length",
            "petal_width",
        ]
#Split the data into training and testing sets
        train_data, test_data = train_test_split(
            data,
            test_size=0.2,
            random_state=42,
            stratify=data["target"],
        )

        train_data = train_data.copy()
        test_data = test_data.copy()

        # Learn mean and standard deviation from training data only to Normalize the features through standardization
        scaler = StandardScaler()
        train_data[features] = scaler.fit_transform(
            train_data[features]
        )

        # Apply the same scaling to the test data.
        test_data[features] = scaler.transform(
            test_data[features]
        )

        folder = Path(clean_path).parent

        train_data.to_csv(folder / "train.csv", index=False) #save the processed dataset
        test_data.to_csv(folder / "test.csv", index=False)

        # Combine both sets into one processed CSV.
        # The split column identifies which rows belong to each set.
        processed_data = pd.concat(
            [
                train_data.assign(split="train"),
                test_data.assign(split="test"),
            ],
            ignore_index=True,
        )

        processed_data.to_csv(
            folder / "processed_iris.csv",
            index=False,
        )

        joblib.dump(scaler, folder / "scaler.joblib")

        print(f"Training rows: {len(train_data)}")
        print(f"Testing rows: {len(test_data)}")
        print(f"Processed files saved in: {folder}")

        return {
            "train": str(folder / "train.csv"),
            "test": str(folder / "test.csv"),
        }


    # Task 4: Train a logistic regression model.
    @task
    def train_model(paths):
        import joblib
        import pandas as pd
        from sklearn.linear_model import LogisticRegression

        train_data = pd.read_csv(paths["train"])

        features = [
            "sepal_length",
            "sepal_width",
            "petal_length",
            "petal_width",
        ]

        X_train = train_data[features]
        y_train = train_data["target"]

        model = LogisticRegression(
            max_iter=1000,
            random_state=42,
        )
        model.fit(X_train, y_train)

        model_path = Path(paths["train"]).with_name("model.joblib")
        joblib.dump(model, model_path)

        print("Model training completed.")
        return str(model_path)

    # Task 5: Evaluate the model.
    @task
    def evaluate_model(model_path, paths):
        import joblib
        import pandas as pd
        from sklearn.metrics import accuracy_score

        test_data = pd.read_csv(paths["test"])

        features = [
            "sepal_length",
            "sepal_width",
            "petal_length",
            "petal_width",
        ]

        X_test = test_data[features]
        y_test = test_data["target"]

        model = joblib.load(model_path)
        predictions = model.predict(X_test)

        accuracy = float(accuracy_score(y_test, predictions))
        print(f"Test accuracy: {accuracy:.2%}")

        return accuracy

    # Connect all five tasks.
    raw_path = load_data()
    clean_path = clean_data(raw_path)
    processed_paths = split_normalize_save(clean_path)
    model_path = train_model(processed_paths)
    evaluate_model(model_path, processed_paths)


# Register the DAG with Airflow.
iris_preprocessing()


Overwriting /content/airflow/dags/iris_preprocessing_dag.py


In [13]:
os.environ["AIRFLOW_HOME"] = "/content/airflow"
os.environ["AIRFLOW__CORE__DAGS_FOLDER"] = "/content/airflow/dags"

# Initialize or update Airflow's metadata database.
!airflow db migrate

2026-10-11T16:32:24.279303Z [info     ] Performing upgrade to the metadata database [airflow.cli.commands.db_command] loc=db_command.py:134 url=sqlite:////content/airflow/airflow.db
2026-10-11T16:32:25.126530Z [info     ] Context impl SQLiteImpl.       [alembic.runtime.migration] loc=migration.py:205
2026-10-11T16:32:25.126845Z [info     ] Will assume non-transactional DDL. [alembic.runtime.migration] loc=migration.py:208
2026-10-11T16:32:26.852219Z [info     ] Context impl SQLiteImpl.       [alembic.runtime.migration] loc=migration.py:205
2026-10-11T16:32:26.852501Z [info     ] Will assume non-transactional DDL. [alembic.runtime.migration] loc=migration.py:208
2026-10-11T16:32:26.853174Z [info     ] Creating Airflow database tables from the ORM [airflow.utils.db] loc=db.py:749
2026-10-11T16:32:26.853415Z [info     ] Creating global lock context   [airflow.utils.db] loc=db.py:730
2026-10-11T16:32:26.853619Z [info     ] Pool status: Pool size: 5  Connections in pool: 0 Current Overflow:

In [14]:
# Execute all tasks in the DAG locally.
!airflow dags test iris_preprocessing

2026-10-11T16:33:00.079751Z [info     ] DAG bundles loaded: dags-folder, example_dags, apache-airflow-providers-common-sql-example-dags, apache-airflow-providers-standard-example-dags [airflow.dag_processing.bundles.manager.DagBundlesManager] loc=manager.py:304
2026-10-11T16:33:00.309161Z [info     ] Added new DAG bundle dags-folder to the database [airflow.dag_processing.bundles.manager.DagBundlesManager] loc=manager.py:378
2026-10-11T16:33:00.309680Z [info     ] Added new DAG bundle example_dags to the database [airflow.dag_processing.bundles.manager.DagBundlesManager] loc=manager.py:378
2026-10-11T16:33:00.310017Z [info     ] Added new DAG bundle apache-airflow-providers-common-sql-example-dags to the database [airflow.dag_processing.bundles.manager.DagBundlesManager] loc=manager.py:378
2026-10-11T16:33:00.310343Z [info     ] Added new DAG bundle apache-airflow-providers-standard-example-dags to the database [airflow.dag_processing.bundles.manager.DagBundlesManager] loc=manager.py:3

Based on teh output, the model loaded a total of 150 rows, of which 120 was used for training nad 30 was reserved for testing. Of teh testing data, the model correctly calssified 28 out of the 30 test dataset.
The airflow acheived an accuracy scode of 93.33%